# xLSTM Exercise Recognition on EgoExo-Fitness Dataset
### Capstone Project: AI-Powered Fitness Coach

This notebook trains the xLSTM temporal model on the EgoExo-Fitness dataset for exercise classification and quality prediction.

## Architecture
```
Video → Frame Sampling → Pose Extraction (MediaPipe) → Chebyshev Interpolation → xLSTM → Heads
                                                                                ↓
                                              ┌──────────────────┬──────────────────┐
                                              ↓                  ↓
                                       Classification      Quality Score
                                       (5 exercises)         (0-5 scale)
```

## Setup Instructions
1. Set your HuggingFace token in Colab secrets: `HF_TOKEN`
2. Connect to a GPU runtime (Runtime → Change runtime type → GPU)
3. Run all cells sequentially

## Step 1: Environment Setup

In [ ]:
#@title Install Dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers huggingface_hub
!pip install mediapipe opencv-python-headless
!pip install pandas tqdm scikit-learn
!pip install tensorboard

import os
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
#@title Authenticate with HuggingFace
from huggingface_hub import login
from google.colab import userdata

# Get HF token from Colab secrets
HF_TOKEN = userdata.get('HF_TOKEN')

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✓ Logged in to HuggingFace")
else:
    print("⚠ HF_TOKEN not found in secrets. Please add it:")
    print("  1. Click the key icon 🔑 in the left sidebar")
    print("  2. Add new secret: HF_TOKEN=<your_token>")
    print("  3. Re-run this cell")

## Step 2: Download Datasets

In [ ]:
#@title Clone Project Repository
import os

WORKSPACE = "/content/fitness-coach"
os.makedirs(WORKSPACE, exist_ok=True)
%cd {WORKSPACE}

# Clone your repository (replace with your actual repo)
!git clone https://github.com/elinelkonyan/fitness-coach-capstone-1.git .

print(f"✓ Cloned to {WORKSPACE}")

In [ ]:
#@title Download EgoExo-Fitness Dataset
from huggingface_hub import snapshot_download
import os

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

print("Downloading EgoExo-Fitness dataset...")
print("  This may take 10-30 minutes depending on your connection")

# Download the dataset
egoexo_path = snapshot_download(
    repo_id="ego-exo/egoexo-fitness",
    repo_type="dataset",
    local_dir=os.path.join(DATA_DIR, "egoexo-fitness"),
    token=HF_TOKEN,
    ignore_patterns=["*.git*", "README.md"]  # Skip unnecessary files
)

print(f"✓ EgoExo-Fitness downloaded to: {egoexo_path}")

# List downloaded files
!ls -la {egoexo_path}

In [ ]:
#@title Download Riccio Dataset (if available on HF)
from huggingface_hub import snapshot_download, list_repo_files
import os

# Check if Riccio is available on HuggingFace
try:
    files = list_repo_files("your-username/riccio-dataset", repo_type="dataset", token=HF_TOKEN)
    print(f"✓ Riccio dataset found: {len(files)} files")
    
    riccio_path = snapshot_download(
        repo_id="your-username/riccio-dataset",
        repo_type="dataset",
        local_dir=os.path.join(DATA_DIR, "riccio"),
        token=HF_TOKEN
    )
    print(f"✓ Riccio downloaded to: {riccio_path}")
except Exception as e:
    print(f"⚠ Riccio dataset not available on HF: {e}")
    print("  Will use EgoExo-Fitness only")
    riccio_path = None

## Step 3: Preprocess - Extract Pose Features

In [ ]:
#@title Extract Pose Features with MediaPipe
import sys
sys.path.insert(0, WORKSPACE)

from fitness_coach.preprocessing.pose_extractor import MediaPipePoseExtractor, batch_extract_videos
import pandas as pd
from pathlib import Path

# Create output directory for features
FEATURE_DIR = "/content/data/features"
os.makedirs(FEATURE_DIR, exist_ok=True)

# Load EgoExo metadata
egoexo_meta = pd.read_csv(os.path.join(DATA_DIR, "egoexo-fitness", "egoexo_fitness_index_split.csv"))
print(f"✓ Loaded EgoExo metadata: {len(egoexo_meta)} samples")
print(f"  Exercises: {egoexo_meta['exercise_class'].nunique()}")
print(f"  Split distribution: {egoexo_meta['split'].value_counts().to_dict()}")

# For smoke test, use only first 20 videos
SMOKE_TEST = True  # Set to False for full training
if SMOKE_TEST:
    egoexo_meta = egoexo_meta.head(20)
    print(f"  Using {len(egoexo_meta)} videos for smoke test")

In [ ]:
#@title Process Videos and Extract Features
import cv2

# Note: EgoExo videos may need to be downloaded separately
# This is a placeholder - adjust based on actual dataset structure

video_paths = []
for idx, row in egoexo_meta.iterrows():
    # Adjust path based on actual dataset structure
    video_path = os.path.join(DATA_DIR, "egoexo-fitness", row.get('video_path', ''))
    if os.path.exists(video_path):
        video_paths.append(video_path)
    else:
        print(f"⚠ Video not found: {video_path}")

print(f"✓ Found {len(video_paths)} videos")

if len(video_paths) > 0:
    # Extract features
    extractor = MediaPipePoseExtractor()
    results = batch_extract_videos(
        video_paths[:10],  # Process first 10 for smoke test
        FEATURE_DIR,
        target_frames=60
    )
    extractor.close()
else:
    print("⚠ No videos found. Check dataset download.")
    print("  EgoExo-Fitness may require additional download steps.")

## Step 4: Train xLSTM Model

In [ ]:
#@title Run Training
import subprocess

OUTPUT_DIR = "/content/results/xlstm_egoexo"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training command
cmd = [
    "python", "train_xlstm_exercise.py",
    "--data-csv", os.path.join(DATA_DIR, "egoexo-fitness", "egoexo_fitness_index_split.csv"),
    "--feature-dir", FEATURE_DIR,
    "--feature-type", "pose",
    "--target-frames", "60",
    "--interpolation", "chebyshev",
    "--epochs", "50" if SMOKE_TEST else "100",
    "--batch-size", "32",
    "--lr", "0.0005",
    "--hidden-size", "128",
    "--num-layers", "2",
    "--output-dir", OUTPUT_DIR,
]

print("Running xLSTM training...")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Epochs: {'50 (smoke)' if SMOKE_TEST else '100'}")

result = subprocess.run(cmd, cwd=WORKSPACE)

if result.returncode == 0:
    print("✓ Training complete!")
else:
    print("✗ Training failed. Check logs above.")

In [ ]:
#@title View Training Results
import json
import matplotlib.pyplot as plt

# Load training history
history_file = os.path.join(OUTPUT_DIR, "training_history.json")
if os.path.exists(history_file):
    with open(history_file) as f:
        history = json.load(f)
    
    # Plot training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.plot(history['val_loss'], label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(history['train_acc'], label='Train Acc')
    ax2.plot(history['val_acc'], label='Val Acc')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Validation Accuracy')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Final Validation Accuracy: {history['val_acc'][-1]:.4f}")
else:
    print("⚠ Training history not found")

In [ ]:
#@title Load Test Results
test_file = os.path.join(OUTPUT_DIR, "test_results.json")
if os.path.exists(test_file):
    with open(test_file) as f:
        test_results = json.load(f)
    
    print("="*60)
    print("TEST RESULTS")
    print("="*60)
    print(f"Test Accuracy: {test_results.get('test_accuracy', 0):.4f}")
    print("\nPer-Class Metrics:")
    for cls_name, metrics in test_results.get('per_class_metrics', {}).items():
        print(f"  {cls_name}: acc={metrics.get('accuracy', 0):.4f} (n={metrics.get('count', 0)})")
else:
    print("⚠ Test results not found")

## Step 5: Save Model to Drive

In [ ]:
#@title Copy Results to Google Drive
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Create output folder
DRIVE_OUTPUT = "/content/drive/MyDrive/fitness-coach/xlstm_results"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy model checkpoint
model_file = os.path.join(OUTPUT_DIR, "xlstm_best.pt")
if os.path.exists(model_file):
    shutil.copy(model_file, DRIVE_OUTPUT)
    print(f"✓ Model saved to: {os.path.join(DRIVE_OUTPUT, 'xlstm_best.pt')}")

# Copy class mapping
class_map = os.path.join(OUTPUT_DIR, "class_map.json")
if os.path.exists(class_map):
    shutil.copy(class_map, DRIVE_OUTPUT)
    print(f"✓ Class map saved to: {os.path.join(DRIVE_OUTPUT, 'class_map.json')}")

# Copy test results
shutil.copy(test_file, DRIVE_OUTPUT)
print(f"✓ Test results saved to: {os.path.join(DRIVE_OUTPUT, 'test_results.json')}")

## Next Steps

1. **Download the model** from your Google Drive
2. **Run inference** using `inference_xlstm_complete.py`
3. **Add Gemma feedback** by running with `--use-gemma` flag

```bash
python inference_xlstm_complete.py \
    --video test_video.mp4 \
    --model-path xlstm_best.pt \
    --use-gemma
```